In [ ]:
from azure.storage.filedatalake import DataLakeServiceClient
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
from io import BytesIO

account_name = "tumorimages60104758"
account_key = "<account key>" #replced with a placeholder

service_client = DataLakeServiceClient(
    account_url=f"https://{account_name}.dfs.core.windows.net",
    credential=account_key
)

raw_container = "raw"
raw_folder = "tumor_images/"

file_system_client = service_client.get_file_system_client(raw_container)
paths = file_system_client.get_paths(path=raw_folder)

data_list = []

for path in paths:
    if path.name.endswith((".jpeg", ".jpg")):
        file_client = file_system_client.get_file_client(path.name)
        download = file_client.download_file()
        file_bytes = download.readall()

        label = "yes" if "/yes/" in path.name.lower() else "no"

        feature1 = len(file_bytes)

        data_list.append({
            "filename": path.name,
            "label": label,
            "feature1": feature1
        })

df = pd.DataFrame(data_list)

silver_container = "silver"
output_path = "features_v1/features.parquet"

silver_client = service_client.get_file_system_client(silver_container)
file_client = silver_client.get_file_client(output_path)

table = pa.Table.from_pandas(df)
pq_bytes = BytesIO()
pq.write_table(table, pq_bytes)
pq_bytes.seek(0)

file_client.upload_data(pq_bytes.read(), overwrite=True)

print("Parquet successfully created in silver/features_v1!")


Parquet successfully created in silver/features_v1!


In [6]:
import pandas as pd
import pyarrow.parquet as pq
from azure.storage.filedatalake import DataLakeServiceClient
from io import BytesIO


silver_container = "silver"
parquet_path = "features_v1/features.parquet"

silver_client = service_client.get_file_system_client(silver_container)
file_client = silver_client.get_file_client(parquet_path)


download = file_client.download_file()
file_bytes = download.readall()

df = pq.read_table(BytesIO(file_bytes)).to_pandas()


In [7]:
# Preview first few rows
print(df.head())

# Check column names
print("Columns:", df.columns.tolist())

# Check number of rows
print("Number of rows:", len(df))


                    filename label  feature1
0  tumor_images/no/1 no.jpeg    no     54521
1  tumor_images/no/10 no.jpg    no      3848
2  tumor_images/no/11 no.jpg    no      3475
3  tumor_images/no/12 no.jpg    no      4142
4  tumor_images/no/13 no.jpg    no      4570
Columns: ['filename', 'label', 'feature1']
Number of rows: 177


In [ ]:
if "label" in df.columns:
    print("Labels column exists.")
    print(df['label'].value_counts())  
else:
    print("Labels column is missing. You need to add it before Phase 2.")


Labels column exists.
no     91
yes    86
Name: label, dtype: int64


In [9]:
print("Missing values per column:\n", df.isnull().sum())


Missing values per column:
 filename    0
label       0
feature1    0
dtype: int64
